# PennyLane Device Selection

This notebook demonstrates how to select different PennyLane devices
when creating an executor via the `Executor` factory. The `backend`
configuration accepts either a **string** (device name) or a ready-made
`qml.devices.Device` instance.

In [11]:
import warnings

import pennylane as qml

from executor import Executor, QuantumCircuit, QuantumOperator

## 1. Default device (`default.qubit`)

When no backend is specified, the factory-created executor falls back to
PennyLane's `default.qubit` simulator.

In [12]:
executor_default = Executor.create("pennylane")
print("Device:", executor_default.device_name)

Device: default.qubit


## 2. String backend - `default.mixed`

Pass the backend device name as a string via `backend=...`.
Any additional `**kwargs` are forwarded to `qml.device()`.

In [13]:
executor_mixed = Executor.create("pennylane", backend="default.mixed")
print("Device:", executor_mixed.device_name)

Device: default.mixed


## 3. Comparing expectation values across devices

A simple Bell-state circuit computed on both devices.
The noiseless results should agree.

In [14]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

op = QuantumOperator(["ZZ"], [1.0])

result_default = executor_default.expectation_value(qc, op)
result_mixed = executor_mixed.expectation_value(qc, op)

print(f"default.qubit: {result_default:.6f}")
print(f"default.mixed: {result_mixed:.6f}")
print(f"Difference:    {abs(result_default - result_mixed):.2e}")

default.qubit: 1.000000
default.mixed: 1.000000
Difference:    0.00e+00


## 4. Sampling with a string device

In [15]:
executor_sample = Executor.create(
    "pennylane",
    backend="default.mixed",
    shots=1000,
    seed=42,
)

qc_sample = QuantumCircuit(2)
qc_sample.h(0)
qc_sample.cx(0, 1)

samples = executor_sample.sample(qc_sample)
print("Samples:", samples)

Samples: [{'11': 497, '00': 503}]


c:\Users\mow\Documents\Dev\Executor\.venv\Lib\site-packages\pennylane\devices\device_api.py:201: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


## 5. Passing a backend instance via the factory

Instead of a string you can pass a pre-configured
`qml.devices.Device` object to `Executor.create(...)`. The executor uses it as-is
and never recreates it internally.

In [16]:
dev = qml.device("default.qubit", wires=4)
executor_custom = Executor.create(dev)

print("Device name: ", executor_custom.device_name)
print("Custom device:", executor_custom._custom_device)
print("Same object:  ", executor_custom._device is dev)

Device name:  default.qubit
Custom device: True
Same object:   True


In [17]:
result_custom = executor_custom.expectation_value(qc, op)
print(f"Expectation value (device instance): {result_custom:.6f}")

Expectation value (device instance): 1.000000


## 6. Config / shots conflict warning

If `shots` is set as an executor parameter **and** a `config`
containing a `shots` value is passed via `**kwargs`, the config
value takes precedence and a `UserWarning` is emitted.

> **Note:** `config` is forwarded to `qml.device()` and must be a
> valid `pennylane.Configuration` object. Here we catch the
> downstream error to focus on the warning.

In [18]:
# --- dict config with 'shots' key ---
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    try:
        Executor.create(
            "pennylane",
            backend="default.qubit",
            shots=100,
            config={"shots": 500},
        )
    except Exception:
        pass  # qml.device() rejects a plain dict config

for w in caught:
    print(f"[{w.category.__name__}] {w.message}")

[UserWarning] The 'shots' parameter (100) is overridden by the shots value (500) from the provided config.


In [19]:
# --- object config with .shots attribute ---
class _FakeConfig:
    shots = 200


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    try:
        Executor.create(
            "pennylane",
            backend="default.qubit",
            shots=100,
            config=_FakeConfig(),
        )
    except Exception:
        pass

for w in caught:
    print(f"[{w.category.__name__}] {w.message}")

[UserWarning] The 'shots' parameter (100) is overridden by the shots value (200) from the provided config.


## 7. Backend instance rejects extra arguments

When a backend *instance* is passed, extra `*args` / `**kwargs`
raise a `TypeError` - the backend is already fully configured.

In [20]:
dev = qml.device("default.qubit", wires=2)
try:
    Executor.create(dev, custom_decomps={})
except TypeError as exc:
    print(f"TypeError: {exc}")

TypeError: Extra positional or keyword arguments are not accepted when 'backend' is a Device instance. Configure the device before passing it to PennyLaneExecutor.
